# Task 2: Biomaker Embedding

This task is about giving biological meaning to the biomarkers (glycans) we previously discovered by embedding them (i.e. converting each glycan into a numerical representation) into a space that reflects their biochemical, functional, and clinical relationships (i.e. such that similar glycans are placed close together in the embedding space based on these features).

In this part, we will:
- Learn a meaningful embedding space that captures relationships between glycans based on structure, origin, tissue, disease, and protein interactions.
- Validate the embedding by using the N-glycans (known structures) as a ground-truth set
- Embed our discovered glycans into this space and assess their closeness to other structures to draw conclusions as to their nature.

In [160]:
import pandas as pd
import numpy as np
import copy
import matplotlib.pyplot as plt
import time
import importlib
import os

# Machine Learning methods
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
from tokenizers import BertWordPieceTokenizer
from transformers import (
    BertTokenizerFast,
    BertConfig,
    BertForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)



# Personal helpers
import helpers_TASK2
importlib.reload(helpers_TASK2)
from helpers_TASK2 import *

In [145]:
# Load the data
glycan_list = pd.read_csv("./data/glycan_embedding/glycan_list.csv")
df_glycan = pd.read_pickle("./data/glycan_embedding/df_glycan.pkl")
glycan_binding = pd.read_pickle("./data/glycan_embedding/glycan_binding.pkl")
N_glycans_df = pd.read_pickle("./data/glycan_embedding/N_glycans_df.pkl")

# Print the shape of each dataset
print(f"Shape of Glycan List (our discovered molecules): {glycan_list.shape}")
print(f"Shape of Glycan Dataset (sequences, species, tissue, disease): {df_glycan.shape}")
print(f"Shape of Protein-Glycan binding interactions Dataset: {glycan_binding.shape}")
print(f"Shape of N-glycans Dataset (N-gylcans sequences for control representation): {N_glycans_df.shape}")

Shape of Glycan List (our discovered molecules): (5, 4)
Shape of Glycan Dataset (sequences, species, tissue, disease): (50589, 23)
Shape of Protein-Glycan binding interactions Dataset: (1465, 2745)
Shape of N-glycans Dataset (N-gylcans sequences for control representation): (48, 24)


In [146]:
# Missing Values
print(f"Glycan List:\n{glycan_list.isna().sum()}\n")
print(f"Glycan DF:\n{df_glycan.isna().sum()}\n")
print(f"Glycan Binding:\n{glycan_binding.isna().sum()}\n")
print(f"N-Glycans:\n{N_glycans_df.isna().sum()}")

Glycan List:
glycan            0
Composition       0
tissue_species    0
tissue_sample     0
dtype: int64

Glycan DF:
glycan                     0
Species                    0
Genus                      0
Family                     0
Order                      0
Class                      0
Phylum                     0
Kingdom                    0
Domain                     0
ref                        0
glytoucan_id           15573
glycan_type            23628
disease_association        0
disease_id                 0
disease_sample             0
disease_direction          0
disease_ref                0
disease_species            0
tissue_sample              0
tissue_id                  0
tissue_ref                 0
tissue_species             0
Composition                0
dtype: int64

Glycan Binding:
3-Anhydro-Gal(a1-3)Gal(b1-4)3-Anhydro-Gal(a1-3)Gal4S                                                                  1461
3-Anhydro-Gal(a1-3)Gal4S(b1-4)3-Anhydro-Gal(a1-3)Gal4S        

## Part 1: Learn a Glycan Embedding Space based on features of glycans of interest (`glycan_list`)

In this part, we will create an embedding space for the glycan libraries that captures meaningful relationships between the features of our glycans of interest, i.e. the columns of `glycan_list`:
- Sequence
- Composition
- Tissue Sample
- Tissue Species

The final goal of this part is to obtain an embedding matrix, where:
- Rows: glycans
- Columns: features (sequence, composition, tissue and species)

For this purpose, we need to find a way to represent the different features appropriately. We will investigate several ways to do this.
1. For Sequence: Using TF-IDF, Counts, or ???
2. For Composition: Using counts or ???
3. For Species: Using counts or ???
4. For Tissues: Using counts or ???

Once we obtain this representation matrix, we will apply different reducing methods to learn a meaningful embedding space. The glycans we use for the learning of the embedding space are in `df_glycans`. We will investigate several methods:
1. PCA
2. SVD
3. ???
4. ???

Finally, when the embedding space is learned, we will evaluate its quality by assessing the closeness of the N-glycans in the new embedding space.

## Part 2: Embed our glycans
Once we find a satisfying embedding space, we will embed the glycans in our `glycan_list`, and identify the *glycans* closest to our glycans of interest.

## Part 3: Enrich our glycans
The next step will be to enrich the embedding space, i.e. including more features in the representation matrix (like associated disease and protein-glycan binding interactions), and using them also to learn the embedding space. We will then identify the glycans closest to the *glycans* we found in the previous part, in order to get a distribution of information on our glycans of interest (e.g., distribution of the most likely diseases the glycan is associated to.)

## Part 1: Learn a Glycan Embedding Space based on features of glycans of interest

In part 1 of our analysis, we will build a feature space that places glycans near each other if they:
- Have similar sequences
- Have similar composition
- Come from similar species
- Come from similar tissues

To do this, we will use machine learning to learn and validate the embedding, starting with simpler, interpretable models and eventually scaling-up to more complex architectures.

In [147]:
# Load df_glycan as is done in glycowork
df_glycan2 = copy.deepcopy(df_glycan)
df_glycan2.set_index("glycan", inplace = True)
#df_glycan2.head(1).style.set_properties(**{'font-size': '8pt', 'font-family': 'Helvetica','border-collapse': 'collapse','border': '1px solid black'})

### Glycan Sequence

The first step is to find a meaningful way to represent the glycan sequences, i.e. learning an embedding space for glycan sequences, where closer glycans have similar sequences.

For this purpose, we will treat each glycan sequence as a "sentence" of glycoletters. We will try two embedding techniques:
- Count vectorizer: this converts a collection of text to a matrix of token counts. It counts the frequency of each glycoletter in a glycan sequence. It captures presence and abundance of substructures, and represents a simple and efficient baseline.
- TF-IDF vectorizer: this converts a collection of text to a matrix of TF-IDF features. It assigns a weight to each glycoletter based on its frequency in a glycan, and how rare it is across all gylcans. It emphasizes rare but distinctive glycoletters, providing discriminative embeddings that reduce noise from common, less informative tokens.

### Glycan Composition

A meaningful way to create an embedding for glycan composition is by transforming each glycan into a vector of monosaccharide counts. It first identifies all unique monosaccharides across the dataset, then represents each glycan as a vector indicating how many times each monosaccharide appears in its composition.

### Glycan Tissue and Species Sample Origin

Two meaningful ways to embed glycan tissue and species sample origin are:
- One-Hot Encoding of Tissue Origin: it captures a binary association between each glycan and each tissue/species. Each row is a glycan, and each column represents a specific tissue type/species. Value = 1 if the glycan was found in that tissue/species, 0 otherwise.
- TF-IDF representation of tissue/species origin: it treats tissue/species annotations like words in a focument. TF-IDF emphasizes rare but informative tissues/species over frequent, less distinguishing ones. Each glycan gets a weighted vector, where values reflect how uniquely the glycan is associated with each tissue/species. This adds nuance, making tissues/species that are common across glycans get lower weight, and rare ones have higher discriminative power.

### Dimensionality Reduction

After representing each glycan feature (e.g., Sequence, Composition, Tissue, Species) as high-dimensional vectors, we optionally apply dimensionality reduction to simplify and structure the embedding space. Each method offers different benefits:

- PCA: Finds new axes (principal components) that capture the most variance in the data. It is useful to help filter out less informative features. It is good for dense data like One-Hot or TF-IDF matrices with many correlated features.
- SVD: Decomposes the input into orthogonal vectors, focusing on capturing dominant patterns, particularly effective for sparse matrices. It is ideal for TF-IDF, Count or One-Hot data, which can be sparse and high-dimensional.
- Nothing: Keeps the full original representation. This retains complete information, avoiding information loss from compression, especially for representations that are not necessarily high-dimensional.

### Trying out and evaluating different combinations of embedding methods

Now, we will combine all these embedding methods, and evaluate them using silhouette score and nearest-neighbor purity. This is the method we follow for evaluating embedding methods: 

We systematically evaluate multiple embedding configurations for glycan data by combining different feature extraction methods (e.g., TF-IDF, One-Hot, Counts) and dimensionality reduction techniques (PCA, SVD) for four types of features: Sequence, Composition, Species, and Tissue.

We explore all combinations of:
- 4 methods for Sequence,
- 2 for Composition,
- 3 for Species,
- 3 for Tissue
→ resulting in 84 total embedding configurations.

For each configuration, we:
1. Generate embeddings for each feature using the chosen method.
2. Reduce dimensionality (if applicable).
3. Concatenate all embeddings into one combined feature space.
4. Evaluate the quality of the embedding using two metrics:
5. Silhouette Score
6. Nearest Neighbor Purity (NN Purity)
7. Record runtime for each configuration.

The Evaluation Metrics we use are Silhouette Score and NN Purity:
1. Silhouette Score
- Measures how well-separated and cohesive the clusters are in the embedding space.
- A value near 1 indicates glycans are well-clustered and distinct from other clusters.
- A value near 0 suggests overlapping clusters or ambiguous placement.

2. Nearest Neighbor (NN) Purity
- For each glycan, we find its k nearest neighbors (here, k=5).
- NN Purity = fraction of neighbors that are in the same class (e.g., N-glycan).
- High purity means local neighborhoods are consistent, i.e., similar glycans are close in the embedding space.

In [162]:
embedding_names = []
feature_dicts = []

# Define options for each feature
sequence_methods = [('TF-IDF', 'SVD'), ('TF-IDF', 'PCA'), ('TF-IDF', None), ('Counts', 'SVD'), ('Counts', 'PCA'), ('Counts', None)]
composition_methods = [('One-Hot', None)]
species_methods = [('One-Hot', None)]
tissue_methods = [('One-Hot', None)]

# Loop through all combinations
for seq_method in sequence_methods:
    for comp_method in composition_methods:
        for species_method in species_methods:
            for tissue_method in tissue_methods:
                
                # Build feature dictionary
                feature_dict = {
                    'Sequence': list(seq_method),
                    'Composition': list(comp_method),
                    'Species': list(species_method),
                    'Tissue': list(tissue_method),
                }
                
                # Create a readable name
                name = f"Sequence_{seq_method[0]}-{seq_method[1]}_Composition_{comp_method[0]}-{comp_method[1]}_Species_{species_method[0]}-{species_method[1]}_Tissue_{tissue_method[0]}-{tissue_method[1]}"
                
                # Save
                feature_dicts.append(feature_dict)
                embedding_names.append(name)

summary_df1 = compare_embeddings(embedding_names, feature_dicts ,df_glycan2, N_glycans_df)
summary_df1

Embedding name: Sequence_TF-IDF-SVD_Composition_One-Hot-None_Species_One-Hot-None_Tissue_One-Hot-None
Features used: {'Sequence': ['TF-IDF', 'SVD'], 'Composition': ['One-Hot', None], 'Species': ['One-Hot', None], 'Tissue': ['One-Hot', None]}
Sequence START
Shape of Matrix: (50589, 2008)
TF-IDF DONE
Shape of Matrix: (50589, 2008)
Shape of Matrix: (50589, 50)
SVD DONE
Shape of Matrix: (50589, 50)
Sequence DONE
Composition START
Shape of Matrix: (50589, 15)
One-Hot DONE
Shape of Matrix: (50589, 15)
Shape of Matrix: (50589, 15)
Composition DONE
Species START
Shape of Matrix: (50589, 268)
One-Hot DONE
Shape of Matrix: (50589, 268)
Shape of Matrix: (50589, 268)
Species DONE
Tissue START
Shape of Matrix: (50589, 262)
One-Hot DONE
Shape of Matrix: (50589, 262)
Shape of Matrix: (50589, 262)
Tissue DONE
Shape of combined embeddings before reduction: (50589, 595)


KeyboardInterrupt: 

I tried using some other methods:
- "GlyBERT": Fine‑tune a BERT‑style Transformer on a large corpus of glycan sequences.
- "Word2Vec": Treat each glycan as a “sentence” of overlapping k‑mer motifs (e.g. tri‑ or tetra‑saccharide substructures), then train a Word2Vec or FastText model (via gensim) to learn dense motif vectors.

After trying to implement these methods, many bugs occured, and it seemed that there was not enough time to try and implement these methods. As a result, I decided to use the best embedding space I found so far, even though it is not optimal.

The results suggest that the best embedding method is:
- Sequence: Counts-SVD
- Composition :One-Hot-None
- Species: One-Hot-None
- Tissue: One-Hot-None 

This configuration achieved the highest scores among all tested combinations with:

- Silhouette Score: 0.389 --> measures how similar each glycan is to its own cluster compared to other clusters. This score suggests a moderate cluster structure, indicating the embedding space captures meaningful groupings of glycans (i.e. N-glycan vs others).
- NN Purity: 0.158 --> measures how often a glycan's k nearest neighbors (in our case 5) belong to the same class. This scores means that on average, ~16% of the neighbors share the same label, which is relatively low. It suggests that local structure is partially preserved, but there is room to improve.
- Time: 8.44 seconds --> efficient and scalable method.

This embedding setup effectively balances biological signal retention (moderate silhouette, best NN purity in its group) with computational efficiency. The choice of Counts over TF-IDF for sequence appears to better preserve structural similarities between glycans, especially when paired with SVD.

## Part 2: Embed our glycans
After finding a satisfying embedding space, we will embed the glycans in our `glycan_list`, and identify the *glycans* closest to our glycans of interest.

Best embedding space:
- Sequence: Counts vectorizer with SVD dimensionality reduction.
- Composition: One-Hot encoding without dimensionality reduction.
- Tissue: One-Hot encoding without dimensionality reduction.
- Species: One-Hot encoding without dimensionality reduction.

In [163]:
sequence_embedding, time_embedding = counts_embedding('glycan', glycan_list)

Counts DONE
Shape of Matrix: (5, 14)


In [ ]:
import ast

# Convert composition strings into dictionaries
glycan_list['Composition_dict'] = glycan_list['Composition'].apply(ast.literal_eval)

# Get all unique monosaccharies across the dataset
monosaccharides = set()
for composition in glycan_list['Composition_dict']:
    monosaccharides.update(composition.keys())

# Create a DataFrame with monosaccharide counts
composition_df = pd.DataFrame([
    {mono: composition.get(mono, 0) for mono in monosaccharides}
    for composition in glycan_list['Composition_dict']
])

---
Now that we have combined the features: sequence proximity, composition, tissue and species, we will add this following feature:
- Disease Association

---
Now that we have combined the features: sequence proximity, composition, tissue, species, and disease assocation, we will add this following feature: 
- Protein-Glycan Binding

---

In [ ]:
glycan_binding['3-Anhydro-Gal(a1-3)Gal(b1-4)3-Anhydro-Gal(a1-3)Gal4S'].dropna()

237     0.010219
273    -0.423532
533    -0.310640
1379   -0.075725
Name: 3-Anhydro-Gal(a1-3)Gal(b1-4)3-Anhydro-Gal(a1-3)Gal4S, dtype: float64

In [ ]:
glycan_binding.shape

(1465, 2745)

In [ ]:
glycan_binding.head()

,3-Anhydro-Gal(a1-3)Gal(b1-4)3-Anhydro-Gal(a1-3)Gal4S,3-Anhydro-Gal(a1-3)Gal4S(b1-4)3-Anhydro-Gal(a1-3)Gal4S,3-Anhydro-Gal(a1-3)Gal4S(b1-4)3-Anhydro-Gal(a1-3)Gal4S(b1-4)3-Anhydro-Gal(a1-3)Gal4S,3-Anhydro-Gal(a1-3)Gal4S(b1-4)3-Anhydro-Gal(a1-3)Gal4S(b1-4)3-Anhydro-Gal(a1-3)Gal4S(b1-4)3-Anhydro-Gal(a1-3)Gal4S,3-Anhydro-Gal(a1-3)Gal4S(b1-4)3-Anhydro-Gal2S(a1-3)Gal4S(b1-4)3-Anhydro-Gal(a1-3)Gal4S,3dGal(b1-3)[Fuc(a1-4)]Glc,3dGal(b1-4)Glc,4d8dNeu5Ac(a2-3)Gal(b1-4)Glc,4dNeu5Ac(a2-3)Gal(b1-4)Glc,7dNeu5Ac(a2-3)Gal(b1-4)Glc,...,wwwSflexneri5c,wwwSflexneriO2c,wwwSflexneriO5c,wwwSisomicin,wwwSmix,wwwTobramycin,wwwTyrS,wwwpHGGs,target,protein
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AADSIPSISPTGIITPTPTQSGMVSNCNKFYDVHSNDGCSAIASSQ...,TAL6-4LysM
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AAFFSLVVLLALLPFGIHASALPSTELTPRVNPNLPGPNDVFVGFR...,rCnSL-proA
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AANEADYQAKLTAYQTELARVQKANADAKAAYEAAVAANNAANAAL...,AntigenI/IIA3VP1
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AASKLGVPQPAQRDQVNCQLYAVQPNDNCIDISSKNNITYAQLLSW...,TAL6-6LysM
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACNNEWEDEQYEQYISFKSPIPAGGEGVTDIYVRYKEDGKVTYRLP...,SP15308A-bot-339-19-339


So we:
- Learn embedding space based on sequences, composition, tissue and species.
- Represent glycans from glycan_list in this space.
- Identify close glycans to each glycan in this space.
- Enrich the glycans with context like: diseases associated with nearby glycans and known protein-binding partners.

In [ ]:
glycan_list

,glycan,Composition,tissue_species,tissue_sample
0,Fuc(a1-?)GlcNAc(b1-2)Man(a1-6)[GlcNAc(b1-2)Man...,"{'dHex': 2, 'HexNAc': 4, 'Hex': 3}",['Homo_sapiens'],['blood']
1,Neu5Ac(a2-?)Gal(b1-4)GlcNAc(b1-2)Man(a1-6)[Glc...,"{'Neu5Ac': 1, 'Hex': 4, 'HexNAc': 4, 'dHex': 1}",['Homo_sapiens'],['blood']
2,Neu5Ac(a2-6)Gal(b1-4)GlcNAc(b1-2)Man(a1-6)[Gal...,"{'Neu5Ac': 1, 'Hex': 5, 'HexNAc': 4}",['Homo_sapiens'],['blood']
3,Neu5Ac(a2-6)Gal(b1-4)GlcNAc(b1-2)Man(a1-6)[Glc...,"{'Neu5Ac': 1, 'Hex': 4, 'HexNAc': 4}",['Homo_sapiens'],['blood']
4,Fuc(a1-2)[GalNAc(a1-3)]Gal(b1-4)GlcNAc(b1-2)Ma...,"{'dHex': 1, 'HexNAc': 5, 'Hex': 5}",['Homo_sapiens'],['blood']


# TODO
- Try BERT for sequence embedding
- Reorganize to be able to apply PCA on individual feature (e.g. species), TF-IDF on individual feature (including species, tissues, etc.)
- Embed glycans in this space
etc.